# 02 · Modelado + Política de decisión

Spec: `docs/superpowers/specs/2026-10-17-modelado-decision-design.md`. El notebook **solo lee**
los resultados de `uv run poe train` (`models/champion_metadata.json`, `models/candidates.joblib`)
y dibuja; no entrena nada. Usa la partición **valid**; las cifras de **test** salen de la metadata
(el test se evaluó una sola vez, al final del entrenamiento). Solo agregados y gráficos, sin filas
de datos.

In [1]:
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_curve

from credit_risk.config import load_config
from credit_risk.data.schema import FEATURES
from credit_risk.data.split import load_partition
from credit_risk.decision.policy import THRESHOLD_GRID
from credit_risk.eda.plots import save_figure
from credit_risk.evaluation.metrics import expected_cost
from credit_risk.features.treatment import drop_underage

cfg = load_config()
FIG_DIR = cfg.paths.reports / "figures"
meta = json.loads((cfg.paths.models / "champion_metadata.json").read_text(encoding="utf-8"))
models = joblib.load(cfg.paths.models / "candidates.joblib")

valid = drop_underage(load_partition("valid", cfg.paths.interim))
X_valid, y_valid = valid[FEATURES], valid[cfg.target]
proba = {name: model.predict_proba(X_valid)[:, 1] for name, model in models.items()}
champion = meta["champion"]
t_star, t_low = meta["thresholds"]["t_star"], meta["thresholds"]["t_low"]
print(f"Campeón: {champion} | valid: {len(valid):,} filas")
print(f"t* = {t_star:.3f} | t_low = {t_low:.3f}")

Campeón: lightgbm | valid: 23,903 filas
t* = 0.140 | t_low = 0.073


## 1. Tratamiento aplicado

Todo se ajusta con **train** y se aplica igual a valid, test y la API (transformadores de
`credit_risk.features.treatment`).

| # | Tema | Tratamiento |
|---|---|---|
| U1 | Duplicados contradictorios | Se conservan |
| U2 | Códigos 96/98 en atrasos | Indicador `has_special_past_due_code`; los conteos 96/98 pasan a 0 |
| U3 | `MonthlyIncome` faltante | Mediana por decil de edad × dependientes, con indicador y respaldo |
| U4 | `RevolvingUtilizationOfUnsecuredLines` | Recorte en el p99 de train |
| U5 | `DebtRatio` | Sin ingreso: ratio = monto ÷ ingreso imputado; luego recorte en el p99 de train |
| C1 | `MonthlyIncome = 0` | Se trata como no disponible, igual que el nulo |
| C2 | Indicadores | Un solo `income_unavailable` (nulo o 0) |
| C3 | `NumberOfDependents` faltante | Se imputa 0 |
| C4 | Grupos de imputación | Decil de edad × dependientes `0,1,2,3,4+`; grupo < 30 filas → mediana del decil → global |
| C5 | Ingreso extremo | Recorte en el p99 de train antes de medianas y ratios |
| C6 | `age < 18` | Se eliminan de las tres particiones |
| C7 | Conteos con colas largas | Recorte en el p99 de train solo en la rama logística |
| C8 | Escala | `log1p` en ingreso y ratios + estandarización, solo en la logística |

In [2]:
preprocessor = models["random_forest"].calibrated_classifiers_[0].estimator[0]
treated = preprocessor.transform(X_valid)
summary = pd.DataFrame(
    {
        "antes": {
            "MonthlyIncome nulo (%)": X_valid["MonthlyIncome"].isna().mean() * 100,
            "MonthlyIncome = 0 (%)": (X_valid["MonthlyIncome"] == 0).mean() * 100,
            "NumberOfDependents nulo (%)": X_valid["NumberOfDependents"].isna().mean() * 100,
        },
        "después": {
            "MonthlyIncome nulo (%)": treated["MonthlyIncome"].isna().mean() * 100,
            "MonthlyIncome = 0 (%)": (treated["MonthlyIncome"] == 0).mean() * 100,
            "NumberOfDependents nulo (%)": treated["NumberOfDependents"].isna().mean() * 100,
        },
    }
)
display(summary.round(2))
print(f"has_special_past_due_code = 1: {treated['has_special_past_due_code'].mean():.2%}")
print(f"income_unavailable = 1: {treated['income_unavailable'].mean():.2%}")

,antes,después
MonthlyIncome nulo (%),19.63,0.0
MonthlyIncome = 0 (%),0.99,0.0
NumberOfDependents nulo (%),2.57,0.0


has_special_past_due_code = 1: 0.16%
income_unavailable = 1: 20.62%


## 2. Comparación de candidatos (valid)

In [3]:
rows = []
for name, c in meta["candidates"].items():
    rows.append(
        {
            "modelo": name,
            "AUC CV": f"{c['cv_auc_mean']:.4f} ± {c['cv_auc_std']:.4f}",
            **{k: round(v, 4) for k, v in c["valid_metrics"].items()},
            "costo esperado": round(c["valid_expected_cost"], 4),
            "t*": c["valid_t_star"],
            "búsqueda (s)": round(c["search_seconds"], 1),
            "latencia (ms)": round(c["latency_ms"], 1),
        }
    )
display(pd.DataFrame(rows).set_index("modelo"))
print("Mejores hiperparámetros:")
for name, c in meta["candidates"].items():
    print(f"- {name}: {c['best_params']}")

,AUC CV,roc_auc,gini,ks,pr_auc,brier,costo esperado,t*,búsqueda (s),latencia (ms)
modelo,,,,,,,,,,
logistic,0.8534 ± 0.0039,0.8557,0.7113,0.5469,0.3873,0.0498,0.2219,0.180,35.1,49.5
random_forest,0.8650 ± 0.0041,0.8660,0.7319,0.5715,0.4052,0.0492,0.2160,0.175,202.6,447.7
lightgbm,0.8653 ± 0.0037,0.8681,0.7362,0.5790,0.4093,0.0490,0.2141,0.140,239.9,93.9


Mejores hiperparámetros:
- logistic: {'model__C': 0.008260808399079604, 'model__l1_ratio': 1.0}
- random_forest: {'model__max_depth': 11, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 40, 'model__n_estimators': 202}
- lightgbm: {'model__colsample_bytree': 0.9160702162124823, 'model__learning_rate': 0.040360819412796234, 'model__min_child_samples': 60, 'model__n_estimators': 227, 'model__num_leaves': 21, 'model__reg_lambda': 4.250192888948996, 'model__subsample': 0.7797802696552814}


In [4]:
fig, ax = plt.subplots(figsize=(6, 5))
for name, p in proba.items():
    fpr, tpr, _ = roc_curve(y_valid, p)
    auc = meta["candidates"][name]["valid_metrics"]["roc_auc"]
    ax.plot(fpr, tpr, label=f"{name} (AUC {auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=0.8)
ax.set_xlabel("Tasa de falsos positivos")
ax.set_ylabel("Tasa de verdaderos positivos")
ax.set_title("Curvas ROC en validación")
ax.legend()
_ = save_figure(fig, FIG_DIR / "roc_valid.png")

## 3. Calibración (valid)

In [5]:
fig, ax = plt.subplots(figsize=(6, 5))
for name, p in proba.items():
    frac_pos, mean_pred = calibration_curve(y_valid, p, n_bins=10, strategy="quantile")
    ax.plot(mean_pred, frac_pos, marker="o", label=name)
ax.plot([0, 0.6], [0, 0.6], "k--", lw=0.8, label="perfecta")
ax.set_xlabel("Probabilidad predicha (media por bin)")
ax.set_ylabel("Tasa de mora observada")
ax.set_title("Curvas de calibración en validación")
ax.legend()
_ = save_figure(fig, FIG_DIR / "calibration_valid.png")
brier = pd.DataFrame(
    {
        "Brier calibrado": {n: c["valid_metrics"]["brier"] for n, c in meta["candidates"].items()},
        "Brier sin calibrar": {
            n: c["valid_metrics_uncalibrated"]["brier"] for n, c in meta["candidates"].items()
        },
    }
)
display(brier.round(5))

,Brier calibrado,Brier sin calibrar
logistic,0.04980,0.05009
random_forest,0.04919,0.04930
lightgbm,0.04904,0.04909


## 4. Costo esperado frente al umbral (campeón, valid)

In [6]:
fn, fp = cfg.costs.false_negative, cfg.costs.false_positive
costs = [expected_cost(y_valid, proba[champion], t, fn, fp) for t in THRESHOLD_GRID]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(THRESHOLD_GRID, costs)
ax.axvline(t_star, color="C3", label=f"t* = {t_star:.3f}")
ax.axvline(fp / (fp + fn), color="C2", ls="--", label=f"FP/(FP+FN) = {fp / (fp + fn):.3f}")
ax.set_xlabel("Umbral de rechazo")
ax.set_ylabel("Costo esperado por cliente")
ax.set_title(f"Costo esperado vs umbral ({champion})")
ax.legend()
_ = save_figure(fig, FIG_DIR / "cost_vs_threshold.png")

## 5. Política de 3 bandas (valid)

In [7]:
print(f"aprobar: p < {t_low:.3f} | revisión: {t_low:.3f} ≤ p < {t_star:.3f}")
print(f"rechazar: p ≥ {t_star:.3f}")
display(pd.DataFrame(meta["valid_bands"]).T.rename_axis("banda"))
print("Sensibilidad al costo FN:FP")
display(pd.DataFrame(meta["cost_sensitivity"]).round(4))
print("Sensibilidad a la capacidad de revisión")
display(pd.DataFrame(meta["capacity_sensitivity"]).round(4))

aprobar: p < 0.073 | revisión: 0.073 ≤ p < 0.140
rechazar: p ≥ 0.140


,rows,share,default_rate
banda,,,
approve,18419.0,0.770573,0.020848
review,2395.0,0.100197,0.092276
reject,3089.0,0.129231,0.322434


Sensibilidad al costo FN:FP


,fn_fp_ratio,t_star,t_low,approval_rate,default_rate_approved,expected_cost
0,3.0,0.280,0.1051,0.9339,0.0410,0.1522
1,5.0,0.140,0.0725,0.8708,0.0291,0.2141
2,10.0,0.100,0.0590,0.8254,0.0250,0.3349
3,20.0,0.045,0.0273,0.6710,0.0145,0.4658


Sensibilidad a la capacidad de revisión


,capacity,t_low,approve_rows,approve_default_rate,review_rows,review_default_rate,reject_rows,reject_default_rate
0,0.05,0.0973,19619,0.0248,1195,0.0987,3089,0.3224
1,0.10,0.0725,18419,0.0208,2395,0.0923,3089,0.3224
2,0.20,0.0450,16031,0.0144,4783,0.0782,3089,0.3224


## 6. Importancia global

Árboles → SHAP (`TreeExplainer`, muestra de 2.000 filas de valid ya transformadas); logística →
coeficientes estandarizados. El estimador base sale de `calibrated_classifiers_[0].estimator`.

In [8]:
base = models[champion].calibrated_classifiers_[0].estimator
features_t = base[:-1].transform(X_valid)
if champion == "logistic":
    importance = pd.Series(base["model"].coef_[0], index=features_t.columns).sort_values()
    title = "Coeficientes estandarizados (logística)"
else:
    import warnings

    sample = features_t.sample(n=min(2000, len(features_t)), random_state=cfg.seed)
    # Silencia avisos informativos de SHAP (importación y formato de salida en LightGBM);
    # además imprimirían rutas locales del entorno en el notebook versionado.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        import shap

        values = shap.TreeExplainer(base["model"]).shap_values(sample)
    values = values[1] if isinstance(values, list) else np.asarray(values)
    if values.ndim == 3:
        values = values[:, :, 1]
    importance = pd.Series(np.abs(values).mean(axis=0), index=sample.columns).sort_values()
    title = f"Importancia SHAP media absoluta ({champion})"
fig, ax = plt.subplots(figsize=(7, 5))
importance.plot.barh(ax=ax)
ax.set_title(title)
_ = save_figure(fig, FIG_DIR / "importance_champion.png")
display(importance.sort_values(ascending=False).round(4).to_frame("importancia"))

,importancia
RevolvingUtilizationOfUnsecuredLines,0.6913
NumberOfTime30-59DaysPastDueNotWorse,0.2897
age,0.2392
NumberOfTimes90DaysLate,0.2076
DebtRatio,0.1701
NumberOfOpenCreditLinesAndLoans,0.1449
NumberOfTime60-89DaysPastDueNotWorse,0.1190
NumberRealEstateLoansOrLines,0.0872
MonthlyIncome,0.0686
income_unavailable,0.0466


## 7. Resultado en test frente a valid

In [9]:
valid_metrics = meta["candidates"][champion]["valid_metrics"]
comparison = pd.DataFrame({"valid": valid_metrics, "test": meta["test_metrics"]})
comparison.loc["costo esperado (t*)"] = [
    meta["candidates"][champion]["valid_expected_cost"],
    meta["test_expected_cost"],
]
display(comparison.round(4))
print("Bandas en test (umbrales fijados en valid)")
display(pd.DataFrame(meta["test_bands"]).T.rename_axis("banda"))

,valid,test
roc_auc,0.8681,0.8650
gini,0.7362,0.7301
ks,0.5790,0.5891
pr_auc,0.4093,0.4010
brier,0.0490,0.0488
costo esperado (t*),0.2141,0.2156


Bandas en test (umbrales fijados en valid)


,rows,share,default_rate
banda,,,
approve,22861.0,0.765119,0.019072
review,3020.0,0.101074,0.100993
reject,3998.0,0.133806,0.315408
